# Onward Subsequential Transducer Inference Algorithm (OSTIA)

This notebook demonstrates the **OSTIA algorithm**, a state-merging algorithm for learning subsequential finite-state transducers from input-output pairs.

## Algorithm Overview

OSTIA (Onward Subsequential Transducers Inference Algorithm) learns a subsequential transducer by:

1. **Building a prefix tree transducer** from input-output examples
2. **Iteratively merging states** that have compatible output behavior
3. **Producing a minimal transducer** that maps inputs to outputs

Unlike RPNI which learns acceptors (with binary accept/reject), OSTIA learns transducers that can produce structured output sequences for each input.

In [13]:
from itertools import count
from typing import Sequence, cast

from state_merging.algorithms.ostia import ostia
from state_merging.automata.SFST import SFST, assert_SFST, run
from state_merging.collections.dense_int_dict import DenseIntDict
from state_merging.collections.dense_int_set import DenseIntSet

Define training data: input-output pairs as both lists and strings. This demonstrates our implementation's type parametricity—the same algorithm learns correctly whether outputs are sequences of integers or characters.

In [14]:
input_set: set[str] = {'a', 'b'}

dataset_list: list[tuple[list[str], list[int]]] = [
    (['a'], [1]),
    (['b'], [1]),
    (['a', 'a'], [0, 1]),
    (['a', 'a', 'a'], [0, 0, 1]),
    (['a', 'b', 'a', 'b'], [0, 1, 0, 1]),
    (['a', 'b'], [0, 1])
]

dataset_str: list[tuple[str, str]] = [
    ('a', '1'),
    ('b', '1'),
    ('aa', '01'),
    ('aaa', '001'),
    ('abab', '0101'),
    ('ab', '01')
]

Learn a transducer with list outputs. The OSTIA algorithm constructs an FST that maps input sequences to lists of integers, using list concatenation as the output operation.

In [15]:
fst_list: SFST[int, str, Sequence[int]]
fst_list, _ = ostia(
    input_set=input_set,
    dataset=dataset_list,
    epsilon=[],
    concat=lambda v1, v2: cast(list[int], v1) + cast(list[int], v2),
    choose_transition=lambda _, trs: next(iter(trs)),
    search_iter=lambda _, qs: qs,
    state_supply=count(),
    empty_fst_state_set=DenseIntSet(),
    empty_transition_mapping={},
    empty_final_output_mapping=DenseIntDict(),
    make_empty_visited_state_set=lambda qs: DenseIntSet(size=len(qs)),
    make_default_populated_mapping=lambda qs: DenseIntDict(size=len(qs), init=list),
    verbose=True
)

assert_SFST(fst_list)

for input, output in dataset_list:
    match run(fst_list, input, cast(list[int], []), lambda v1, v2: v1 + cast(list[int], v2)):
        case None:
            assert False, \
                f"learned FST rejected positive data {input, output}"
        case _, real_output:
            assert real_output == output, \
                f"learned FST incorrectly output {real_output} on positive data {input, output}"

learning from the following positive data by state-merging: [
	['a'], [1]
	['b'], [1]
	['a', 'a'], [0, 1]
	['a', 'a', 'a'], [0, 0, 1]
	['a', 'b', 'a', 'b'], [0, 1, 0, 1]
	['a', 'b'], [0, 1]
]

naively constructed PTT:
SFST(state_set=DenseIntSet({0, 1, 2, 3, 4, 5, 6, 7}), input_set={'a', 'b'}, initial_state=0, transitions={(0, 'a'): (1, []), (0, 'b'): (2, []), (1, 'a'): (3, []), (3, 'a'): (4, []), (1, 'b'): (5, []), (5, 'a'): (6, []), (6, 'b'): (7, [])}, initial_output=[], final_outputs=DenseIntDict({1: [1], 2: [1], 3: [0, 1], 4: [0, 0, 1], 5: [0, 1], 7: [0, 1, 0, 1]}))

onwardized PTT:
SFST(state_set=DenseIntSet({0, 1, 2, 3, 4, 5, 6, 7}), input_set={'a', 'b'}, initial_state=0, transitions={(0, 'a'): (1, []), (0, 'b'): (2, [1]), (1, 'a'): (3, [0]), (3, 'a'): (4, [0, 1]), (1, 'b'): (5, [0, 1]), (5, 'a'): (6, [0, 1]), (6, 'b'): (7, [])}, initial_output=[], final_outputs=DenseIntDict({1: [1], 2: [], 3: [1], 4: [], 5: [], 7: []}))

merge candidate: 1 into 0
redirecting state 0 on input a to

Learn a transducer with string outputs. Same OSTIA algorithm, different output type.

In [16]:
fst_str: SFST[int, str, Sequence[str]]
fst_str, _ = ostia(
    input_set=input_set,
    dataset=dataset_str,
    epsilon='',
    concat=lambda v1, v2: cast(str, v1) + cast(str, v2),
    choose_transition=lambda _, trs: next(iter(trs)),
    search_iter=lambda _, qs: iter(qs),
    state_supply=count(),
    empty_fst_state_set=DenseIntSet(),
    empty_transition_mapping={},
    empty_final_output_mapping=DenseIntDict(),
    make_empty_visited_state_set=lambda qs: DenseIntSet(size=len(qs)),
    make_default_populated_mapping=lambda qs: DenseIntDict(size=len(qs), init=list),
    verbose=True
)

assert_SFST(fst_str)

for input, output in dataset_str:
    match run(fst_str, input, '', lambda v1, v2: v1 + cast(str, v2)):
        case None:
            assert False, \
                f"learned FST rejected positive data {input, output}"
        case _, real_output:
            assert real_output == output, \
                f"learned FST incorrectly output {real_output} on positive data {input, output}"

learning from the following positive data by state-merging: [
	a, 1
	b, 1
	aa, 01
	aaa, 001
	abab, 0101
	ab, 01
]

naively constructed PTT:
SFST(state_set=DenseIntSet({0, 1, 2, 3, 4, 5, 6, 7}), input_set={'a', 'b'}, initial_state=0, transitions={(0, 'a'): (1, ''), (0, 'b'): (2, ''), (1, 'a'): (3, ''), (3, 'a'): (4, ''), (1, 'b'): (5, ''), (5, 'a'): (6, ''), (6, 'b'): (7, '')}, initial_output='', final_outputs=DenseIntDict({1: '1', 2: '1', 3: '01', 4: '001', 5: '01', 7: '0101'}))

onwardized PTT:
SFST(state_set=DenseIntSet({0, 1, 2, 3, 4, 5, 6, 7}), input_set={'a', 'b'}, initial_state=0, transitions={(0, 'a'): (1, ''), (0, 'b'): (2, '1'), (1, 'a'): (3, '0'), (3, 'a'): (4, '01'), (1, 'b'): (5, '01'), (5, 'a'): (6, '01'), (6, 'b'): (7, '')}, initial_output='', final_outputs=DenseIntDict({1: '1', 2: '', 3: '1', 4: '', 5: '', 7: ''}))

merge candidate: 1 into 0
redirecting state 0 on input a to state 0
attempting to merge state 1 into state 0
attempting to merge state 3 into state 0
merges 